# SageMaker Workflow :: Stakeholder Question 2

"**Estimate how many days it will take a 311 complaint to close.**"

Use this notebook to review and run a complete SageMaker **Linear Learner** workflow for this stakeholder question. This notebook is designed for the Day 28 in-class compare-and-contrast activity so that you can connect the SageMaker built-in algorithm workflow to the sklearn work you completed previously.

At a high level, this notebook will help you:
- Load a modeling dataset from S3,
- Preprocess the data into the format Linear Learner expects,
- Upload train and test files back to S3,
- Launch a SageMaker training job,
- Use Batch Transform to generate predictions, and
- Evaluate the model output in the notebook.

# Setup

This notebook uses SageMaker's built-in **Linear Learner** algorithm to train a baseline model for this stakeholder question. The overall workflow is different from sklearn: you will still prepare data in the notebook, but the actual training job runs on separately managed compute resources in SageMaker and reads its input data from S3.

In the cells below, you will:
- Import the libraries needed for SageMaker, pandas, and evaluation,
- Connect to your SageMaker session and execution role,
- Define your S3 bucket and folder paths, and
- Retrieve the correct regional container image for Linear Learner.

In [1]:
import io
import boto3
import sagemaker
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
s3client = boto3.client('s3')

print(f"Region: {region}")
print(f"Role: {role}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: us-east-1
Role: arn:aws:iam::471112527784:role/LabRole


## S3 configuration

Update the bucket and prefix below so they match the location of your exported modeling CSV. Keeping a clear folder structure in S3 matters because the training data, test data, model artifacts, and prediction outputs are all written to separate locations rather than staying inside notebook memory as they would in sklearn.

In [22]:
# ---------------------------------------------------------------
# Update these to match your S3 bucket and folder structure
# ---------------------------------------------------------------
S3_BUCKET = 'cmse492-dasarkes-nyc311-471112527784-us-east-1-an'
S3_PREFIX = 'modeling'

LL_IMAGE = sagemaker.image_uris.retrieve('linear-learner', region)

## Helper functions

Linear Learner expects a very specific CSV format: all features must be numeric, the label must be in the first column, and there should be no header row. These helper functions make it easier to upload properly formatted files to S3 and reload outputs such as batch prediction results.

In [23]:
def upload_csv_to_s3(df, bucket, key):
    """Upload a DataFrame as a headerless CSV to S3."""
    buf = io.BytesIO()
    df.to_csv(buf, index=False, header=False)
    buf.seek(0)
    s3client.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"Uploaded s3://{bucket}/{key} with {len(df):,} rows")
    return f"s3://{bucket}/{key}"

def download_csv_from_s3(bucket, key, **kwargs):
    """Download a CSV from S3 into a DataFrame."""
    obj = s3client.get_object(Bucket=bucket, Key=key)
    data = obj['Body'].read().decode('utf-8')
    return pd.read_csv(io.StringIO(data), **kwargs)

def make_ll_dataframe(X, y):
    """
    Combine label (y) and features (X) into the format Linear Learner expects:
    label first, all numeric, no header.
    """
    if hasattr(X, 'toarray'):
        X = X.toarray()
    label_col = pd.Series(y, name='label').reset_index(drop=True)
    feat_df = pd.DataFrame(X).reset_index(drop=True)
    return pd.concat([label_col, feat_df], axis=1)

## Load and preprocess the data

This stakeholder question is a **regression** problem because the target is a continuous outcome: `days_to_close`. The notebook loads the exported modeling CSV from S3, identifies the feature columns, and then one-hot encodes the categorical predictors so that every input to Linear Learner is numeric.

Unlike classification, the label here is not a class indicator but a numeric value. That is why the training configuration later will use `predictor_type='regressor'` and will also normalize the label values during training.

In [24]:
# Update your path to your modeling data for this stakeholder question as needed
df = download_csv_from_s3(S3_BUCKET, f"{S3_PREFIX}/q2_modeling_data.csv")
print(df.shape)
display(df.head())

# Features and target
cat_cols = ['agency', 'borough', 'problem']
num_cols = ['day_of_week', 'hour_of_day', 'same_day_complaint_volume']
target = 'days_to_close'

X = df[cat_cols + num_cols]
y = df[target].astype(float)

# One-hot encode categoricals; pass numerics through
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print(f"Training target mean: {y_train.mean():.2f}")
print(f"Training target std: {y_train.std():.2f}")

(173851, 8)


,agency,borough,problem,incident_zip,day_of_week,hour_of_day,same_day_complaint_volume,days_to_close
0,DCWP,BRONX,Consumer Complaint,10455.0,5,13,19,33
1,DCWP,BROOKLYN,Consumer Complaint,11207.0,5,11,19,4
2,DCWP,BROOKLYN,Consumer Complaint,11234.0,5,3,19,34
3,DCWP,BRONX,Consumer Complaint,10454.0,5,11,19,33
4,DCWP,QUEENS,Consumer Complaint,11423.0,5,15,19,33


Train: (139080, 168) Test: (34771, 168)
Training target mean: 2.06
Training target std: 5.10


## Upload train and test files to S3

Even though you are working inside a notebook, SageMaker's managed training job cannot directly access variables sitting in notebook memory. That is why we convert the encoded arrays into headerless CSV files, place the label first, and upload the train and test data to S3 before training begins.

In [25]:
train_path = upload_csv_to_s3(
    make_ll_dataframe(X_train, y_train),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-regression/train/train.csv",
)

test_path = upload_csv_to_s3(
    make_ll_dataframe(X_test, y_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-regression/test/test.csv",
)

Uploaded s3://cmse492-dasarkes-nyc311-471112527784-us-east-1-an/modeling/ll-regression/train/train.csv with 139,080 rows
Uploaded s3://cmse492-dasarkes-nyc311-471112527784-us-east-1-an/modeling/ll-regression/test/test.csv with 34,771 rows


## Train the Linear Learner model

For this problem, the estimator is configured as a **regressor**. Because the target is continuous, `normalize_label=True` is appropriate here so SageMaker can scale the target values internally during training.

Use the code exactly as written below, reviewing the values that are set. Note that this is one of the main points where the workflow diverges from sklearn's `model.fit(...)` pattern.

In [26]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

ll_model = Estimator(
    image_uri=LL_IMAGE,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-regression/output",
    sagemaker_session=session,
)

ll_model.set_hyperparameters(
    predictor_type='regressor',
    feature_dim=X_train.shape[1],
    mini_batch_size=1000,
    epochs=10,
    normalize_data=True,
    normalize_label=True,
)

ll_model.fit(
    {'train': TrainingInput(train_path, content_type='text/csv')},
    logs=False, # Change this to True if you'd like to see the training output.
)

INFO:sagemaker:Creating training-job with name: linear-learner-2026-04-08-13-46-31-005



2026-04-08 13:46:32 Starting - Starting the training job..
2026-04-08 13:46:48 Starting - Preparing the instances for training...
2026-04-08 13:47:11 Downloading - Downloading input data.........
2026-04-08 13:48:02 Downloading - Downloading the training image...................
2026-04-08 13:49:43 Training - Training image download completed. Training in progress..............
2026-04-08 13:50:53 Uploading - Uploading generated training model..
2026-04-08 13:51:06 Completed - Training job completed


## Run Batch Transform for predictions

Instead of deploying a real-time endpoint, this notebook uses **Batch Transform** to score the full test set. That is a good fit for your experimentation because SageMaker spins up temporary inference compute, writes predictions back to S3, and then shuts the job down when it is finished. This avoids leaving a persistent endpoint running, which would consume resources and create additional costs.

In [27]:
transformer = ll_model.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f"s3://{S3_BUCKET}/{S3_PREFIX}/ll-regression/predictions",
    accept='text/csv',
    assemble_with='Line',
)

# Run on the test set (features only — no label column)
test_features_path = upload_csv_to_s3(
    pd.DataFrame(X_test),
    S3_BUCKET,
    f"{S3_PREFIX}/ll-regression/test/test-features.csv",
)

transformer.transform(
    test_features_path,
    content_type='text/csv',
    split_type='Line',
    wait=True,
)

print('Batch transform complete.')

INFO:sagemaker:Creating model with name: linear-learner-2026-04-08-13-52-03-970
INFO:sagemaker:Creating transform job with name: linear-learner-2026-04-08-13-52-07-455


Uploaded s3://cmse492-dasarkes-nyc311-471112527784-us-east-1-an/modeling/ll-regression/test/test-features.csv with 34,771 rows
................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
[04/08/2026 13:57:26 INFO 140311874410304] Memory profiler is not enabled by the environment variable ENABLE_PROFILER.
/opt/amazon/lib/python3.8/site-packages/mxnet/model.py:97: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if num_device is 1 and 'dist' not in kvstore:
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:495: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if cons['type'] is 'ineq':
/opt/amazon/lib/python3.8/site-packages/scipy/optimize/_shgo.py:743: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if len(self.X_min) is not 0:
[04/08/2026 13:57:29 WARNING 140311874410304] Loggers have already been setup.
[04/08/2026 13:57:30 INFO 140311874410304] loaded entry point class alg

## Evaluate the predictions

The output for the regression model is a single predicted numeric value for each row. To make the results easier to interpret, the evaluation compares Linear Learner against a simple naive baseline that predicts the training-set mean for every case.

In [28]:
# Download predictions and evaluate
pred_key = f"{S3_PREFIX}/ll-regression/predictions/test-features.csv.out"
preds = download_csv_from_s3(S3_BUCKET, pred_key, header=None)
display(preds.head())

y_pred = preds.iloc[:, 0].astype(float)

# Naive mean-prediction baseline for comparison
y_mean_pred = np.full(len(y_test), y_train.mean())

print('=== Regression Results ===')
print(f"{'Metric':<10} {'Naive mean':>14} {'Linear Learner':>16}")
print('-' * 42)
print(f"{'MAE':<10} {mean_absolute_error(y_test, y_mean_pred):>14.2f} "
      f"{mean_absolute_error(y_test, y_pred):>16.2f}")
print(f"{'RMSE':<10} {mean_squared_error(y_test, y_mean_pred)**0.5:>14.2f} "
      f"{mean_squared_error(y_test, y_pred)**0.5:>16.2f}")
print(f"{'R²':<10} {r2_score(y_test, y_mean_pred):>14.4f} "
      f"{r2_score(y_test, y_pred):>16.4f}")

,0
0,-0.195231
1,0.153813
2,3.559988
3,0.045532
4,1.893574


=== Regression Results ===
Metric         Naive mean   Linear Learner
------------------------------------------
MAE                  2.80             1.88
RMSE                 5.21             4.14
R²                -0.0000           0.3671


## Wrap-up

After you run the notebook, compare the results here to the sklearn workflow you used previously. As you review the outputs, pay attention to what changed conceptually: the model training happened as a SageMaker job, the data had to be staged in S3, and Batch Transform handled prediction without creating a persistent endpoint.

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.